# Telco Customer Churn Prediction
End-to-end analysis: feature engineering → model comparison → XGBoost → SHAP explainability.

**Dataset**: IBM Telco Customer Churn (7 043 rows, ~26.5% churn rate)

In [1]:
import sys, pathlib
# Works whether nbconvert runs from repo root or notebooks/
for _p in [pathlib.Path("."), pathlib.Path("..")]:
    if (_p / "src").exists():
        sys.path.insert(0, str(_p.resolve()))
        break

import warnings; warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.features import build_features, get_X_y, get_feature_names
from src.train import compare_models, train_best_model, save_model, load_model
from src.explain import global_importance, local_explanation

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
print("Setup OK")

Setup OK


## 1. Load & Inspect Raw Data

In [2]:
import generate_data

DATA_CSV = pathlib.Path("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
if not DATA_CSV.exists():
    generate_data.main()

df_raw = pd.read_csv(DATA_CSV)
print(f"Shape: {df_raw.shape}")
print(f"Churn rate: {(df_raw['Churn'] == 'Yes').mean():.1%}")
df_raw.head(3)

Wrote 7,043 rows to data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
  Churn rate : 22.6%
  Tenure     : min=1  max=72  median=37
  Monthly $  : min=18.00  max=119.98
Shape: (7043, 21)
Churn rate: 22.6%


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,TID_2462685,Female,0,Yes,Yes,65,No,No phone service,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,29.1700,1907.6500,No
1,TID_2023147,Male,1,Yes,No,51,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,18.0500,929.6000,No
2,TID_8893571,Female,1,No,No,62,Yes,No,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,No,Mailed check,89.3300,5545.4700,No


In [3]:
# Missing values check
missing = df_raw.isnull().sum()
print("Missing values:", missing[missing > 0].to_dict() or "none")
print("\nDtype summary:")
print(df_raw.dtypes.value_counts())

Missing values: none

Dtype summary:
str        17
int64       2
float64     2
Name: count, dtype: int64


## 2. Feature Engineering

Key transformations applied by `build_features()`:
- Binary Yes/No columns → 0/1 (including "No phone/internet service" → 0)
- Categorical columns → one-hot (InternetService, Contract, PaymentMethod)
- Engineered: `num_services`, `avg_monthly_charges`, `charges_increase`, `is_new_customer`, `is_long_term`


In [4]:
df = build_features(df_raw, fit=True)
X, y = get_X_y(df)
feature_names = get_feature_names(df)

print(f"Features: {X.shape[1]}")
print(f"Churn rate after encoding: {y.mean():.1%}")
print("\nSample features:", feature_names[:8])
X.head(3)

Features: 31
Churn rate after encoding: 22.6%

Sample features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity']


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,MonthlyCharges,TotalCharges,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,num_services,avg_monthly_charges,charges_increase,is_new_customer,is_long_term
0,0,0,1,1,65,0,0,0,0,0,0,0,0,1,29.1700,1907.6500,False,False,True,False,False,True,False,False,True,False,0,29.3485,-0.1785,0,1
1,1,1,1,0,51,1,1,0,0,0,0,0,0,1,18.0500,929.6000,False,False,True,False,True,False,False,False,False,True,2,18.2275,-0.1775,0,1
2,0,1,0,0,62,1,0,0,1,0,0,1,0,0,89.3300,5545.4700,False,True,False,True,False,False,False,False,False,True,3,89.4431,-0.1131,0,1


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Train churn: {y_train.mean():.1%}  |  Test churn: {y_test.mean():.1%}")

Train: 5,634  |  Test: 1,409
Train churn: 22.6%  |  Test churn: 22.6%


## 3. Model Comparison (5-Fold Cross-Validation)

Four classifiers benchmarked on ROC-AUC and F1.
Class imbalance handled via `class_weight="balanced"` (sklearn) and `scale_pos_weight=3` (XGBoost).


In [6]:
comparison = compare_models(X_train, y_train, cv_folds=5)
print(comparison.to_string(index=False))

              Model  AUC (mean)  AUC (std)  F1 (mean)  F1 (std)
Logistic Regression      0.7968     0.0200     0.5448    0.0196
      Random Forest      0.7900     0.0195     0.5360    0.0246
  Gradient Boosting      0.7866     0.0167     0.4268    0.0166
            XGBoost      0.7823     0.0155     0.5327    0.0187


## 4. Train Best Model on Full Training Set

In [7]:
result = train_best_model(X_train, y_train, X_test, y_test, model_name="XGBoost")
print(f"AUC       : {result['auc']}")
print(f"F1        : {result['f1']}")
print(f"Precision : {result['precision']}")
print(f"Recall    : {result['recall']}")

AUC       : 0.7802
F1        : 0.5283
Precision : 0.4403
Recall    : 0.6604


## 5. Evaluation Plots

In [8]:
from pathlib import Path
Path("figures").mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC
fpr, tpr = result["roc_curve"]
axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {result['auc']:.3f}")
axes[0].plot([0,1],[0,1],"k--",lw=1)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve"); axes[0].legend()

# Precision-Recall
prec, rec = result["pr_curve"]
axes[1].plot(rec, prec, lw=2, color="darkorange")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")

# Confusion Matrix
cm = result["confusion_matrix"]
im = axes[2].imshow(cm, cmap="Blues")
axes[2].set_xticks([0,1]); axes[2].set_xticklabels(["No Churn","Churn"])
axes[2].set_yticks([0,1]); axes[2].set_yticklabels(["No Churn","Churn"])
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("Actual")
axes[2].set_title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i,j]), ha="center", va="center",
                     color="white" if cm[i,j] > cm.max()/2 else "black")

fig.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.savefig("figures/evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/evaluation_plots.png")

Saved figures/evaluation_plots.png


## 6. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) assigns each feature a contribution to the
prediction for every customer. Mean |SHAP| gives a model-level importance ranking
that is consistent with individual explanations.


In [9]:
imp = global_importance(result["model"], X_test, feature_names)
top15 = imp.head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15["feature"][::-1], top15["mean_abs_shap"][::-1], color="steelblue")
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Top 15 Features by SHAP Importance")
plt.tight_layout()
plt.savefig("figures/shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/shap_importance.png")
print(top15[["feature","mean_abs_shap"]].to_string(index=False))

Saved figures/shap_importance.png
                       feature  mean_abs_shap
                        tenure         0.6120
       Contract_Month-to-month         0.4143
   InternetService_Fiber optic         0.3569
                MonthlyCharges         0.3037
           avg_monthly_charges         0.2760
                OnlineSecurity         0.1487
             Contract_Two year         0.1421
              charges_increase         0.1244
                   TechSupport         0.1009
                  TotalCharges         0.0961
                  is_long_term         0.0856
                        gender         0.0654
                  num_services         0.0640
PaymentMethod_Electronic check         0.0584
                  OnlineBackup         0.0426


## 7. Local Explanation — Single Customer

In [10]:
# Pick a customer with high predicted churn probability
probs = result["model"].predict_proba(X_test)[:, 1]
high_risk_idx = probs.argmax()
X_example = X_test.iloc[[high_risk_idx]]
print(f"Customer #{high_risk_idx}  |  P(churn) = {probs[high_risk_idx]:.3f}")

local_exp = local_explanation(result["model"], X_example, feature_names)
print("\nTop drivers:")
print(local_exp.head(8).to_string(index=False))

Customer #341  |  P(churn) = 0.971



Top drivers:
                       feature    value  shap_value
                MonthlyCharges 117.9800      0.9724
           avg_monthly_charges 117.2984      0.6216
                        tenure       19      0.5557
   InternetService_Fiber optic     True      0.3559
       Contract_Month-to-month     True      0.2766
             Contract_Two year    False      0.1349
                  is_long_term        0      0.1186
PaymentMethod_Electronic check     True      0.1140


## 8. Save Model

In [11]:
path = save_model(result["model"], feature_names)
print(f"Saved → {path}")

# Verify round-trip
model_loaded, names_loaded = load_model()
probs_reloaded = model_loaded.predict_proba(X_test)[:, 1]
assert len(probs_reloaded) == len(X_test)
print("Load verification OK")

Saved → models/churn_model.joblib


Load verification OK


## 9. Key Findings

| Metric | Value |
|--------|-------|
| Test AUC | see cell 4 output |
| Top churn driver | Contract type (Month-to-month) |
| Second driver | Tenure (short tenure → higher risk) |
| Third driver | InternetService_Fiber optic |

**Actionable insights:**
- Target Month-to-month customers with tenure < 12 months
- Prioritise customers without OnlineSecurity + TechSupport
- Electronic check users churn more — nudge toward auto-pay
